In [ ]:
import os, sys, shutil, zipfile, time
from pathlib import Path
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

os.chdir("/kaggle/working")

# 1. Search for and extract any zip archives in /kaggle/input
for z in Path("/kaggle/input").rglob("*.zip"):
    print(f"Extracting {z}...")
    with zipfile.ZipFile(z, "r") as zf:
        zf.extractall("/kaggle/working")

# 2. Robustly find and copy directories from capstone-code if not already extracted
src_candidates = [p for p in Path("/kaggle/input").rglob("src") if (p / "trust").is_dir() or (p / "model").is_dir()]
if src_candidates and not Path("/kaggle/working/src").exists():
    shutil.copytree(src_candidates[0], "/kaggle/working/src")
    print(f"Copied src from {src_candidates[0]}")

configs_candidates = [p for p in Path("/kaggle/input").rglob("configs") if (p / "default.yaml").is_file()]
if configs_candidates and not Path("/kaggle/working/configs").exists():
    shutil.copytree(configs_candidates[0], "/kaggle/working/configs")
    print(f"Copied configs from {configs_candidates[0]}")

tests_candidates = [p for p in Path("/kaggle/input").rglob("tests") if (p / "test_determinism.py").is_file()]
if tests_candidates and not Path("/kaggle/working/tests").exists():
    shutil.copytree(tests_candidates[0], "/kaggle/working/tests")
    print(f"Copied tests from {tests_candidates[0]}")

# Copy metadata, RESULTS.md, pytest.ini
os.makedirs("/kaggle/working/data/metadata", exist_ok=True)
for meta_f in ["feature_stats_dev.json", "label_stats_dev.json"]:
    found = list(Path("/kaggle/input").rglob(meta_f))
    if found:
        shutil.copy2(found[0], f"/kaggle/working/data/metadata/{meta_f}")
        print(f"Copied {meta_f} to /kaggle/working/data/metadata/")

for extra_f in ["RESULTS.md", "pytest.ini"]:
    found = list(Path("/kaggle/input").rglob(extra_f))
    if found:
        shutil.copy2(found[0], f"/kaggle/working/{extra_f}")
        print(f"Copied {extra_f} to /kaggle/working/")

# 3. Symlink dataset dev partitions and processed splits
p_dev = None
for p in Path("/kaggle/input").rglob("client_00.parquet"):
    if "partitions" in str(p) and "dev" in str(p):
        p_dev = p.parent.parent if p.parent.name.startswith("seed_") else p.parent
        break

pr_dev = None
for p in Path("/kaggle/input").rglob("test.parquet"):
    if "processed" in str(p) and "dev" in str(p):
        pr_dev = p.parent
        break

if p_dev:
    os.makedirs("/kaggle/working/data/partitions", exist_ok=True)
    os.system(f"ln -sfn {p_dev} /kaggle/working/data/partitions/dev")
    print(f"Linked partitions/dev -> {p_dev}")

if pr_dev:
    os.makedirs("/kaggle/working/data/processed", exist_ok=True)
    os.system(f"ln -sfn {pr_dev} /kaggle/working/data/processed/dev")
    print(f"Linked processed/dev -> {pr_dev}")

print("Working dir contents:", os.listdir("/kaggle/working"))
assert Path("/kaggle/working/src").exists(), "src directory not found!"
assert Path("/kaggle/working/configs").exists(), "configs directory not found!"
assert Path("/kaggle/working/data/partitions/dev").exists(), "data/partitions/dev not found!"
assert Path("/kaggle/working/data/processed/dev").exists(), "data/processed/dev not found!"
assert Path("/kaggle/working/data/metadata/feature_stats_dev.json").exists(), "data/metadata/feature_stats_dev.json not found!"
print("Setup complete and verified!")


In [ ]:
import os
print("Running core regression test suite...")
ret = os.system("PYTHONPATH=. pytest tests/test_determinism.py tests/test_harness.py tests/test_metrics.py tests/test_variants.py")
print(f"Test suite exit code: {ret}")
assert ret == 0, f"Test suite failed with exit code {ret}"
print("All core regression tests passed successfully!")


In [ ]:
import os, time
print("Launching Phase E4.2c part1: References (Step B) and Mechanism Diagnostics (Step D) [151 configs] with 12 workers on Dual Tesla T4 GPUs...")
t_start = time.time()
ret = os.system("PYTHONPATH=. python src/experiments/run_phase_e4_2c.py --shard part1 --workers 12 --output-dir results/runs/phase_e4_2c_part1")
t_elapsed = time.time() - t_start
print(f"part1 finished in {t_elapsed:.1f}s with exit code {ret}")
assert ret == 0, f"part1 failed with exit code {ret}"


In [ ]:
import os, zipfile
from pathlib import Path
print("Packaging phase_e4_2c_part1_results.zip...")
res_dir = Path("/kaggle/working/results/runs/phase_e4_2c_part1")
assert res_dir.exists(), "Results directory not found!"
zip_path = "/kaggle/working/phase_e4_2c_part1_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(res_dir):
        for file in files:
            fpath = Path(root) / file
            zf.write(fpath, fpath.relative_to(res_dir))
size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"Results successfully packaged: {zip_path} ({size_mb:.2f} MB)")
